In [2]:
import os
from dotenv import load_dotenv
from langchain.chains.retrieval_qa.base import RetrievalQA
from langchain_core import chat_history

load_dotenv("../../.env.aiapikey")
api_key = os.getenv("DoogieOpenaiKey")
os.environ['OPENAI_API_KEY'] = api_key

In [3]:
import openai

#aiclient = openai.OpenAI(api_key=api_key)
aiClient = openai.OpenAI()
print(openai.__version__)

2.8.1


In [ ]:
try:
    modelslist = aiClient.models.list()
    print("✅ 모델 목록 가져오기 성공!")
    for model in sorted(modelslist.data, key=lambda model: model.id):
        print(model.id)

except Exception as e:
    print("❌ 모델 목록 가져오기 실패:")
    print(e)

In [8]:
# step 1-1: creat vstore at openAI,
#
vector_store = aiClient.vector_stores.create(
    name="vstore_soccerrule",
    #document_count=10,
    #embedding_function="text-embedding-ada-002",
    #embedding_dimension=1536,
    #model="text-embedding-ada-002"
)
print("vector_store.id: ", vector_store.id)
print("vector_store.name: ", vector_store.name)
#print("vector_store.model: ", vector_store.model)
#print("vector_store.embedding_dimension: ", vector_store.embedding_dimension)
#print("vector_store.document_count: ", vector_store.document_count)
#print("vector_store.status: ", vector_store.status)
#print("vector_store.error: ", vector_store.error)
#print("vector_store.created_at: ", vector_store.created_at)
#print("vector_store.updated_at: ", vector_store.updated_at)
#print("vector_store.embedding_function: ", vector_store.embedding_function)
#print("vector_store.embedding_dimension: ", vector_store.embedding_dimension)


vector_store.id:  vs_6927ecc0ce70819181a7d1b3d06d0529
vector_store.name:  vstore_soccerrule


In [24]:
# step 1-2: open local file
#
file_stream = open("SoccerRule.pdf", "rb")
file_stream.seek(0, os.SEEK_END)
file_size = file_stream.tell()
file_stream.seek(0)
print( file_stream.name )
print( file_size )

SoccerRule.pdf
22073717


In [16]:
# step 1-3: upload local file to vector_store at openAI
# WARNING!!!: uploaded file be CHARGE !!!!!!!!!!!!
# you cant show file at openAI
#   https://platform.openai.com/storage/
file_batch = aiClient.vector_stores.file_batches.upload_and_poll(
    vector_store_id=vector_store.id,
    files=[file_stream],
)

In [17]:
# step 1-4: check uploaded file
#
print( file_batch.status )
print( file_batch.file_counts)

completed
FileCounts(cancelled=0, completed=1, failed=0, in_progress=0, total=1)


In [21]:
# step 2-1: ready instruction
#
instruction = '''
[목적]
이 GPT는 축구 규칙을 상세히 설명해 주는 챗봇 입니다.

[규칙]
1. 사용자가 축구 규칙에 대해 질문하면 업로드된 파일에서 해당 내용을 찾아 자세히 답변합니다.
2. 파일안에서 마땅한 답을 찾을 수 없거나 축구 규칙에 관한 질문이 아니면 ">>>축구 규칙에 관한 질문만 부탁해요^^" 라고 답해주세요.
3. 답변의 형태는 아래 예시와 같이 해주세요
예시)
-  질문 : 질문 내용
-  답변 : 답변내용 
4. 모든 질문에 한국어로 답변해주세요.
'''

In [25]:
# step 2-2: create Assistants
#
myAssistant = aiClient.beta.assistants.create(
    name = "SoccerRule",
    model="gpt-3.5-turbo-1106",
    instructions=instruction,
    #tools=[{"type": "file", "file_id": file_batch.id}],
    tools = [ {"type" : "file_search"} ],
    tool_resources = { "file_search" : {"vector_store_ids" : [vector_store.id] } },
)
print( myAssistant.id)

asst_aiMRu44vdBfUizSlWBRHp9RX


In [ ]:
# step 3-1: change & update Assistants
# DO NOT EXCUTE !!!
myAssistant = aiClient.beta.assistants.update(
    assistant_id=myAssistant.id,    # your own assistant id
    model="new_other_model",
    instructions=new_other_instruction,
    tools=[{"type": "new_type"}],
    tool_resources = { "file_search" : {"vector_store_ids" : [new_vector_store.id] } },
)

In [38]:
# step 3-2: list Assistants
#
myAssistant_list = aiClient.beta.assistants.list(
    order="desc",
    limit=20,
)
print( len(myAssistant_list.data) )
print( myAssistant_list.data[0])

1
Assistant(id='asst_aiMRu44vdBfUizSlWBRHp9RX', created_at=1764292952, description=None, instructions='\n[목적]\n이 GPT는 축구 규칙을 상세히 설명해 주는 챗봇 입니다.\n\n[규칙]\n1. 사용자가 축구 규칙에 대해 질문하면 업로드된 파일에서 해당 내용을 찾아 자세히 답변합니다.\n2. 파일안에서 마땅한 답을 찾을 수 없거나 축구 규칙에 관한 질문이 아니면 ">>>축구 규칙에 관한 질문만 부탁해요^^" 라고 답해주세요.\n3. 답변의 형태는 아래 예시와 같이 해주세요\n예시)\n-  질문 : 질문 내용\n-  답변 : 답변내용 \n4. 모든 질문에 한국어로 답변해주세요.\n', metadata={}, model='gpt-3.5-turbo-1106', name='SoccerRule', object='assistant', tools=[FileSearchTool(type='file_search', file_search=FileSearch(max_num_results=None, ranking_options=FileSearchRankingOptions(score_threshold=0.0, ranker='default_2024_08_21', hybrid_search=None)))], response_format='auto', temperature=1.0, tool_resources=ToolResources(code_interpreter=None, file_search=ToolResourcesFileSearch(vector_store_ids=['vs_6927ecc0ce70819181a7d1b3d06d0529'])), top_p=1.0, reasoning_effort=None)


In [43]:
# step 3-3: retrieve Assistant
#
myAssistant2 = aiClient.beta.assistants.retrieve(assistant_id=myAssistant.id)
print( myAssistant2.id )
print( myAssistant2.name )
print( myAssistant2.model )
print( myAssistant2.instructions )
print( "TOOL:", myAssistant2.tools )
print( "TOOL_RSC:", myAssistant2.tool_resources )

asst_aiMRu44vdBfUizSlWBRHp9RX
SoccerRule
gpt-3.5-turbo-1106

[목적]
이 GPT는 축구 규칙을 상세히 설명해 주는 챗봇 입니다.

[규칙]
1. 사용자가 축구 규칙에 대해 질문하면 업로드된 파일에서 해당 내용을 찾아 자세히 답변합니다.
2. 파일안에서 마땅한 답을 찾을 수 없거나 축구 규칙에 관한 질문이 아니면 ">>>축구 규칙에 관한 질문만 부탁해요^^" 라고 답해주세요.
3. 답변의 형태는 아래 예시와 같이 해주세요
예시)
-  질문 : 질문 내용
-  답변 : 답변내용 
4. 모든 질문에 한국어로 답변해주세요.

TOOL: [FileSearchTool(type='file_search', file_search=FileSearch(max_num_results=None, ranking_options=FileSearchRankingOptions(score_threshold=0.0, ranker='default_2024_08_21', hybrid_search=None)))]
TOOL_RSC: ToolResources(code_interpreter=None, file_search=ToolResourcesFileSearch(vector_store_ids=['vs_6927ecc0ce70819181a7d1b3d06d0529']))


In [69]:
# step 4-1: create thread
#
myThread = aiClient.beta.threads.create()
print( myThread )

/var/folders/2y/695wr9bj2hl6y2fvhrwrs_4m0000gn/T/ipykernel_44246/1838635403.py:3: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  myThread = aiClient.beta.threads.create()


Thread(id='thread_eAvvh5aofG9qUnlDgJdKByWL', created_at=1764304607, metadata={}, object='thread', tool_resources=ToolResources(code_interpreter=None, file_search=None))


In [70]:
# step 4-2: create thread.message
#
myMessage = aiClient.beta.threads.messages.create(
    thread_id=myThread.id,
    role="user",
    content="축구장 크기는? "
)
print( myMessage )

/var/folders/2y/695wr9bj2hl6y2fvhrwrs_4m0000gn/T/ipykernel_44246/2335881552.py:3: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  myMessage = aiClient.beta.threads.messages.create(


Message(id='msg_5aTcku4dmV95p8BQc8tE3drJ', assistant_id=None, attachments=[], completed_at=None, content=[TextContentBlock(text=Text(annotations=[], value='축구장 크기는? '), type='text')], created_at=1764304614, incomplete_at=None, incomplete_details=None, metadata={}, object='thread.message', role='user', run_id=None, status=None, thread_id='thread_eAvvh5aofG9qUnlDgJdKByWL')


In [71]:
# step 4-3:
#
myThread_list = aiClient.beta.threads.retrieve(
    thread_id=myThread.id
)
print( myThread_list )

/var/folders/2y/695wr9bj2hl6y2fvhrwrs_4m0000gn/T/ipykernel_44246/2516478380.py:3: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  myThread_list = aiClient.beta.threads.retrieve(thread_id=myThread.id)


Thread(id='thread_eAvvh5aofG9qUnlDgJdKByWL', created_at=1764304607, metadata={}, object='thread', tool_resources=ToolResources(code_interpreter=ToolResourcesCodeInterpreter(file_ids=[]), file_search=None))


In [109]:
# step 5-1: create Run
#
myRun = aiClient.beta.threads.runs.create(
    thread_id=myThread.id,
    assistant_id=myAssistant.id,
)

/var/folders/2y/695wr9bj2hl6y2fvhrwrs_4m0000gn/T/ipykernel_44246/1676901093.py:3: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  myRun = aiClient.beta.threads.runs.create(


In [110]:
# step 5-2: execute Run & show Run.status
#
import time

while myRun.status not in [ "completed", "failed" ]:
    myRun = aiClient.beta.threads.runs.retrieve(
        thread_id=myThread.id,
        run_id=myRun.id,
    )
    print( myRun.status )
    time.sleep(1)

/var/folders/2y/695wr9bj2hl6y2fvhrwrs_4m0000gn/T/ipykernel_44246/213412817.py:6: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  myRun = aiClient.beta.threads.runs.retrieve(


in_progress
in_progress
completed


In [111]:
# step 5-3: check result at myThread
#
myMessage_response = aiClient.beta.threads.messages.list(
    thread_id=myThread.id,
)

for each_msg in myMessage_response:
    print( each_msg.run_id, each_msg.role, each_msg.content[0].text.value )
    print("----------")

/var/folders/2y/695wr9bj2hl6y2fvhrwrs_4m0000gn/T/ipykernel_44246/2313054914.py:3: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  myMessage_response = aiClient.beta.threads.messages.list(


run_0C8v1dkrl61RDjaz5JvtADAY assistant 골키퍼는 오프사이드 룰에 적용되지 않습니다. 오프사이드는 볼을 받으려는 공격 선수에만 적용되며, 골키퍼는 이 규칙의 적용을 받지 않습니다. 자세한 내용은 "SoccerRule.pdf"의 2021/22 경기규칙 | 심판을 위한 실전 가이드라인 부분에서 확인할 수 있습니다.
----------
None user 골키퍼는 오프사이트 룰에 해당하는가?
----------
run_r1dOydq2Gk87Sp1HnSVwnbO6 assistant 오프사이드 판정은 볼이 플레이되는 순간을 기준으로 합니다. 이때의 수비수보다 더 가까운 위치에 있을 경우 오프사이드가 선언됩니다. 더 자세한 내용은 "SoccerRule.pdf" 파일에 명시되어 있습니다. 해당 내용을 확인하시려면 파일을 참조해주시기 바랍니다.
----------
run_r1dOydq2Gk87Sp1HnSVwnbO6 assistant 오프사이드 판정은 볼이 플레이되는 순간을 기준으로 하며, 이때의 수비수보다 더 가까운 위치에 있을 경우에 오프사이드가 선언됩니다. 자세한 내용은 "SoccerRule.pdf"의 2021/22 경기규칙 | 심판을 위한 실전 가이드라인 부분에서 확인하실 수 있습니다. 
----------
None user 수비수보다 가깝다는 것이 어디를 기준으로 하는가?
----------
run_9hiflhLgAK0CIRxfYktTwDn3 assistant 오프사이드는 볼을 받으려는 선수가 상대팀 수비수보다 최근에 볼이 나간 때 그 선수의 위치가 더 가까울 때 발생합니다.

오프사이드 판정에 대한 규칙은 "SoccerRule.pdf" 파일에 명시되어 있습니다.  에서 자세한 내용을 확인하실 수 있습니다.
----------
None user 어디보다 가깝다는 거야?
----------
run_tXF7q4AZ3S7lF1thzu8UbXaw assistant 오프사이드 룰은 축구 경기 중 공격수가 수비수보다 공을 받을 때, 볼을 받기 전에 수

In [106]:
# step 5-4: create and add new msg to thread, show threads' msg
#
myMessage = aiClient.beta.threads.messages.create(
    thread_id=myThread.id,
    role="user",
    content="골키퍼는 오프사이트 룰에 해당하는가?"
)
print( myMessage )

myMessage_response = aiClient.beta.threads.messages.list(
    thread_id=myThread.id,
)

for each_msg in myMessage_response:
    print( each_msg.run_id, each_msg.role, each_msg.content[0].text.value )
    print("----------")


/var/folders/2y/695wr9bj2hl6y2fvhrwrs_4m0000gn/T/ipykernel_44246/4030759244.py:3: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  myMessage = aiClient.beta.threads.messages.create(


Message(id='msg_TrFxwgdbszYdFvj7FwD59c2W', assistant_id=None, attachments=[], completed_at=None, content=[TextContentBlock(text=Text(annotations=[], value='골키퍼는 오프사이트 룰에 해당하는가?'), type='text')], created_at=1764306434, incomplete_at=None, incomplete_details=None, metadata={}, object='thread.message', role='user', run_id=None, status=None, thread_id='thread_eAvvh5aofG9qUnlDgJdKByWL')


/var/folders/2y/695wr9bj2hl6y2fvhrwrs_4m0000gn/T/ipykernel_44246/4030759244.py:10: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  myMessage_response = aiClient.beta.threads.messages.list(


None user 골키퍼는 오프사이트 룰에 해당하는가?
----------
run_r1dOydq2Gk87Sp1HnSVwnbO6 assistant 오프사이드 판정은 볼이 플레이되는 순간을 기준으로 합니다. 이때의 수비수보다 더 가까운 위치에 있을 경우 오프사이드가 선언됩니다. 더 자세한 내용은 "SoccerRule.pdf" 파일에 명시되어 있습니다. 해당 내용을 확인하시려면 파일을 참조해주시기 바랍니다.
----------
run_r1dOydq2Gk87Sp1HnSVwnbO6 assistant 오프사이드 판정은 볼이 플레이되는 순간을 기준으로 하며, 이때의 수비수보다 더 가까운 위치에 있을 경우에 오프사이드가 선언됩니다. 자세한 내용은 "SoccerRule.pdf"의 2021/22 경기규칙 | 심판을 위한 실전 가이드라인 부분에서 확인하실 수 있습니다. 
----------
None user 수비수보다 가깝다는 것이 어디를 기준으로 하는가?
----------
run_9hiflhLgAK0CIRxfYktTwDn3 assistant 오프사이드는 볼을 받으려는 선수가 상대팀 수비수보다 최근에 볼이 나간 때 그 선수의 위치가 더 가까울 때 발생합니다.

오프사이드 판정에 대한 규칙은 "SoccerRule.pdf" 파일에 명시되어 있습니다.  에서 자세한 내용을 확인하실 수 있습니다.
----------
None user 어디보다 가깝다는 거야?
----------
run_tXF7q4AZ3S7lF1thzu8UbXaw assistant 오프사이드 룰은 축구 경기 중 공격수가 수비수보다 공을 받을 때, 볼을 받기 전에 수비수보다 더 가까이 있으면 오프사이드가 선언됩니다. 선수는 먼저 공을 받는 선수가 되지 못하며, 이는 경기의 공정성과 골의 쉬운 창출을 막기 위해 도입된 규칙입니다.

더 자세한 내용은 공식 규칙서에 안내되어 있습니다. 이 문제에서 제공된 파일을 통해 자세한 내용을 확인할 수 있습니다. 
----------
None user 오프사이드 룰에 대해 설명해 줘
-

In [21]:
# step 6-1: openAI에 생성되어 작업 내용이 포함되어 있는 Assistant 확인, 제거
#
r1 = aiClient.beta.assistants.list()
print( len(r1.data) )
if ( len(r1.data) > 0 ):
    for re in r.data:
        print( re.id, re.name, re.tools, re.tool_resources )

0


In [18]:
for r1e in r.data:
    response = aiClient.beta.assis.delete( r1e.id )
    print( response )

In [19]:
# step 6-2: openAI에 업로드한 파일들 목록 확인, 제거
#
r2 = aiClient.files.list()
print( len(r2.data) )
if ( len(r2.data) > 0 ):
    for r2e in r2.data:
        #print( r2e )
        print( r2e.id, r2e.purpose, r2e.filename, r2e.bytes )

0


In [20]:
for r2e in r2.data:
    response = aiClient.files.delete( r2e.id )
    print( response )